# NYC Taxi Data Intelligence — NumPy & pandas walkthrough

Explore the Seaborn dataset, then compare this exploratory workflow with the reusable pipeline in `src/taxi_analytics/`.

## Load and inspect

In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
df=sns.load_dataset('taxis')
display(df.head())
print(df.shape)
display(df.dtypes.to_frame('dtype'))
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))

## NumPy arrays and safe arithmetic

In [ ]:
distance=df['distance'].dropna().to_numpy(dtype=float)
print(np.mean(distance),np.median(distance),np.percentile(distance,[25,50,75]))
fare=df['fare'].to_numpy(dtype=float,na_value=np.nan)
tip=df['tip'].to_numpy(dtype=float,na_value=np.nan)
tip_pct=np.divide(tip,fare,out=np.full(tip.shape,np.nan),where=fare>0)*100
print(tip_pct[:10])

## Datetime feature engineering

In [ ]:
work=df.copy()
work['pickup']=pd.to_datetime(work['pickup'],errors='coerce')
work['dropoff']=pd.to_datetime(work['dropoff'],errors='coerce')
work['duration_minutes']=(work['dropoff']-work['pickup']).dt.total_seconds()/60
work['pickup_hour']=work['pickup'].dt.hour
work['weekday']=work['pickup'].dt.day_name()
display(work[['pickup','dropoff','duration_minutes','pickup_hour','weekday']].head())

## Grouped analysis

In [ ]:
display(work.groupby('payment',dropna=False).agg(records=('fare','size'),valid_fares=('fare','count'),mean_fare=('fare','mean'),median_fare=('fare','median')).round(2))

## Correlation and outlier investigation

Correlation is not causation. An IQR flag is a prompt to investigate, not proof of an error.

In [ ]:
display(work.select_dtypes(include='number').corr().round(2))
x=work['fare'].dropna(); q1,q3=x.quantile([.25,.75]); iqr=q3-q1
lo,hi=q1-1.5*iqr,q3+1.5*iqr
print('bounds',lo,hi,'flagged',int(((x<lo)|(x>hi)).sum()))

## Challenges
1. Find the busiest pickup hour.
2. Compare mean and median fare by payment.
3. Investigate missing pickup/drop-off zones.
4. Add a safe ratio feature and test zero denominators.
5. Write three evidence-backed findings and two limitations.